# 🔄 Cycles and Loops in LangGraph

## Learning Objectives
In this notebook, you will learn:
1. **Cyclic Graphs** - How to build loops in LangGraph using conditional edges that route back to earlier nodes
2. **Self-Correcting Agents** - How to generate code, validate it, and automatically retry on failure
3. **Iterative Refinement** - How to deepen research findings across multiple loop iterations
4. **Loop Termination** - How to safely bound loops with iteration counters and max-iteration guards

## Prerequisites
- Familiarity with LangGraph `StateGraph`, nodes, and edges (earlier notebooks in `03_LangGraph_Fundamentals`)
- Understanding of conditional edges / routing functions
- `OPENAI_API_KEY` set in a `.env` file at the project root

---
## 📦 Setup

We import the LangGraph and LangChain building blocks needed for cyclic graphs, load environment
variables from `.env`, and initialize the LLM that both demos in this notebook will share.

In [ ]:
# ============================================================================
# ENVIRONMENT SETUP: Imports and LLM Initialization
# ============================================================================
import operator
from typing import Literal

from typing_extensions import Annotated, TypedDict

from dotenv import load_dotenv

from langchain.chat_models import init_chat_model
from langchain_core.messages import AIMessage, BaseMessage, HumanMessage, SystemMessage
from langgraph.graph import END, START, StateGraph

# Load API keys (OPENAI_API_KEY, etc.) from the project's .env file
load_dotenv()

# Initialize the LLM used by both the self-correcting code generator and the
# iterative research demo below
llm = init_chat_model("gpt-4o-mini", temperature=0.0)

print(f"🤖 LLM initialized: gpt-4o-mini (temperature=0.0)")
print("✅ Environment ready!")

---
## 🔁 Part 1: Self-Correcting Code Generation

A **cycle** in LangGraph is just a conditional edge that routes back to a node the graph has
already visited. Here we use that pattern to build a code generator that writes Python code,
validates it against real test cases, and loops back to regenerate the code whenever validation
fails - stopping either on success or after a maximum number of iterations.

### Key Concepts:
- **Loop-back edge**: A conditional edge whose "not done yet" branch points to an earlier node
- **Guard condition**: An iteration counter (`max_iterations`) that prevents infinite loops
- **State accumulation**: `Annotated[list[str], operator.add]` appends each new error instead of overwriting it

### 1.1 📋 `CodeGenState`

The state tracks the task description, the current code attempt, the running list of validation
errors, how many iterations have run, the iteration cap, and whether validation has succeeded.

In [ ]:
# ============================================================================
# CODEGENSTATE: Graph State for the Self-Correcting Code Generator
# ============================================================================
class CodeGenState(TypedDict):
    task: str
    code: str
    errors: Annotated[list[str], operator.add]  # Accumulates errors across iterations
    iteration: int
    max_iterations: int
    success: bool

### 1.2 🔧 Building the Self-Correcting Graph

`demo_self_correcting_code` wires up three nodes:
- **`generate`** - writes code on the first pass, or fixes the previous attempt using the latest error
- **`validate`** - compiles the code, runs it against test cases, and records any failure as an error
- **`finalize`** - a no-op pass-through reached once validation succeeds or the iteration cap is hit

The conditional edge `should_continue` is what creates the cycle: it routes back to `generate`
when validation fails and iterations remain, or forward to `finalize` otherwise.

In [ ]:
# ============================================================================
# DEMO_SELF_CORRECTING_CODE: Self-Correcting Code Generator
# ============================================================================
"""Self-correcting code generator."""

def generate_code(state: CodeGenState) -> dict:
    if state["iteration"] == 0:
        # First attempt
        prompt = f"Write Python code for: {state['task']}\nReturn only the code."
    else:
        # Correction attempt
        prompt = (
            f"Fix this Python code:\n{state['code']}\n\n"
            f"Errors:\n{state['errors'][-1]}\n\n"
            "Return only the corrected code."
        )
    response = llm.invoke(prompt)
    code = response.content.strip()
    # Clean up markdown code blocks if present
    if code.startswith("```"):
        code = code.split("```")[1]
        if code.startswith("python"):
            code = code[6:]
    return {"code": code, "iteration": state["iteration"] + 1}

def validate_code(state: CodeGenState) -> dict:
    code = state["code"]
    # Step 1: Does it compile?
    try:
        compile(code, "<string>", "exec")
    except SyntaxError as e:
        return {"errors": [f"SyntaxError: {e}"], "success": False}

    # Step 2: Does it RUN and produce correct results?
    test_cases = [
        ([3, 1, 4, 1, 5, 9], 5),  # normal case
        ([1, 1, 1], None),  # all same -> no second largest
        ([7], None),  # single element
        ([3, -1, 3, 5, 5], 3),  # duplicates at top
    ]
    namespace = {}
    try:
        exec(code, namespace)
    except Exception as e:
        return {"errors": [f"Runtime error: {e}"], "success": False}

    if "solve" not in namespace:
        return {"errors": ["Function 'solve' not found in code"], "success": False}

    for inputs, expected in test_cases:
        try:
            result = namespace["solve"](inputs)
            if result != expected:
                return {
                    "errors": [
                        f"solve({inputs}) returned {result}, expected {expected}"
                    ],
                    "success": False,
                }
        except Exception as e:
            return {"errors": [f"solve({inputs}) raised {e}"], "success": False}

    return {"success": True}

def should_continue(state: CodeGenState) -> Literal["generate", "end"]:
    if state["success"]:
        return "end"
    elif state["iteration"] >= state["max_iterations"]:
        return "end"
    else:
        return "generate"

def finalize(state: CodeGenState) -> dict:
    return state

graph = StateGraph(CodeGenState)
graph.add_node("generate", generate_code)
graph.add_node("validate", validate_code)
graph.add_node("finalize", finalize)

graph.add_edge(START, "generate")
graph.add_edge("generate", "validate")
graph.add_conditional_edges(
    "validate", should_continue, {"generate": "generate", "end": "finalize"}
)  # Loop back to "generate" if not successful and under max iterations, otherwise go to "finalize"
graph.add_edge("finalize", END)

app = graph.compile()

# visualize the graph
print("\n--- Mermaid Graph ---")
# print(app.get_graph().draw_mermaid())
# save as PNG
png_bytes = app.get_graph().draw_mermaid_png()
with open("graph_code.png", "wb") as f:
    f.write(png_bytes)
print("\nGraph saved to graph_code.png")

print("Self-Correcting Code Generator:\n")
result = app.invoke(
    {
        "task": "a function that calculates factorial recursively",
        "code": "",
        "errors": [],
        "iteration": 0,
        "max_iterations": 3,
        "success": False,
    }
)

print(f"Task: {result['task']}")
print(f"Iterations: {result['iteration']}")
print(f"Success: {result['success']}")
print(f"Final Code:\n{result['code']}")

---
## 🔬 Part 2: Iterative Research Refinement

The second demo applies the same loop-back pattern to a research workflow: each pass researches
a topic, generates a follow-up question from the latest findings, and - unless the depth cap has
been reached - loops back to research that follow-up question before finally synthesizing
everything into one summary.

### Key Insight:
> A cycle doesn't have to be a "retry on failure" loop. Here the loop condition is simply a depth
> counter, so the same graph shape drives progressively deeper research instead of error correction.

### 2.1 📋 `ResearchState`

Tracks the topic, the accumulated findings and follow-up questions from each round, how deep the
research has gone, the maximum depth allowed, and the final synthesized summary.

In [ ]:
# ============================================================================
# RESEARCHSTATE: Graph State for the Iterative Research Workflow
# ============================================================================
class ResearchState(TypedDict):
    topic: str
    findings: Annotated[list[str], operator.add]  # Accumulates findings across rounds
    questions: list[str]
    iteration: int
    max_depth: int
    summary: str

### 2.2 🔧 Building the Iterative Research Graph

`demo_iterative_research` wires up three nodes:
- **`research`** - answers the topic on the first pass, or digs into the latest follow-up question
- **`generate_questions`** - asks the LLM for one deeper question based on the newest finding
- **`synthesize`** - combines every round of findings into a single coherent summary

The conditional edge `should_continue` creates the cycle: it routes back to `research` while
`iteration` is below `max_depth`, and forward to `synthesize` once the depth cap is reached.

In [ ]:
# ============================================================================
# DEMO_ITERATIVE_RESEARCH: Iterative Research That Goes Deeper Based on Findings
# ============================================================================
def demo_iterative_research():
    """Iterative research that goes deeper based on findings."""

    def research(state: ResearchState) -> dict:
        print(f"\n{'\u2500' * 50}")
        print(f"📚 [RESEARCH] Depth {state['iteration'] + 1}/{state['max_depth']}")
        if state["iteration"] == 0:
            query = f"Give me 3 key facts about: {state['topic']}"
            print(f"   Starting fresh on: {state['topic']}")
        else:
            question = state["questions"][-1] if state["questions"] else "elaborate"
            query = f"Based on these findings:\n{state['findings'][-1]}\n\nGo deeper: {question}"
            print(f"   Following up on: {question}")
        response = llm.invoke(query)
        print(f"   \u2705 Found {len(response.content.splitlines())} lines of findings")
        print(f"   Preview: {response.content[:120]}...")
        return {"findings": [response.content]}

    def generate_questions(state: ResearchState) -> dict:
        print(f"\n{'\u2500' * 50}")
        print(f"🤔 [QUESTIONING] Analyzing latest findings...")
        response = llm.invoke(
            f"Based on this finding:\n{state['findings'][-1]}\n\n"
            "What's one deeper question to explore? Reply with just the question."
        )
        print(f"   Next question: {response.content.strip()}")
        return {"questions": [response.content], "iteration": state["iteration"] + 1}

    def synthesize(state: ResearchState) -> dict:
        print(f"\n{'\u2500' * 50}")
        print(
            f"🧬 [SYNTHESIZE] Combining {len(state['findings'])} rounds of findings..."
        )
        all_findings = "\n\n".join(state["findings"])
        response = llm.invoke(
            f"Synthesize these findings into a coherent summary:\n\n{all_findings}"
        )
        print(f"   \u2705 Summary generated ({len(response.content.split())} words)")
        return {"summary": response.content}

    def should_continue(state: ResearchState) -> Literal["research", "synthesize"]:
        if state["iteration"] >= state["max_depth"]:
            print(
                f"\n🏁 [ROUTER] Max depth reached ({state['iteration']}/{state['max_depth']}) -> synthesizing"
            )
            return "synthesize"
        print(
            f"\n🔄 [ROUTER] Depth {state['iteration']}/{state['max_depth']} -> going deeper"
        )
        return "research"

    graph = StateGraph(ResearchState)
    graph.add_node("research", research)
    graph.add_node("generate_questions", generate_questions)
    graph.add_node("synthesize", synthesize)

    graph.add_edge(START, "research")
    graph.add_edge("research", "generate_questions")
    graph.add_conditional_edges(
        "generate_questions",
        should_continue,
        {"research": "research", "synthesize": "synthesize"},
    )
    graph.add_edge("synthesize", END)

    app = graph.compile()

    print("=" * 50)
    print("🔬 ITERATIVE RESEARCH WORKFLOW")
    print("=" * 50)

    result = app.invoke(
        {
            "topic": "quantum computing applications",
            "findings": [],
            "questions": [],
            "iteration": 0,
            "max_depth": 2,
            "summary": "",
        }
    )

    print(f"\n{'=' * 50}")
    print(f"📊 RESEARCH COMPLETE")
    print(f"   Topic: {result['topic']}")
    print(f"   Depth reached: {result['iteration']}")
    print(f"   Findings collected: {len(result['findings'])}")
    print(f"   Questions explored: {len(result['questions'])}")
    print(f"\n📝 Final Summary:\n{result['summary']}")

---
## ▶️ Running the Demos

The original `__main__` guard is kept verbatim - Jupyter sets `__name__` to `"__main__"`, so this
cell runs as-is. Only one demo runs by default; uncomment the other line to try it instead.

In [ ]:
# ============================================================================
# RUN: Execute One of the Two Demos
# ============================================================================
if __name__ == "__main__":
    # demo_self_correcting_code()
    demo_iterative_research()

---
## 📝 Summary

In this notebook, we learned:

### 1. Cyclic Graphs in LangGraph
- A cycle is a conditional edge whose branches include a route back to an already-visited node
- `add_conditional_edges` with a mapping like `{"generate": "generate", "end": "finalize"}` is what wires up the loop-back
- `Annotated[list[str], operator.add]` lets state fields accumulate values (errors, findings) across iterations instead of being overwritten

### 2. Self-Correcting Code Generation
- `generate -> validate -> (loop back to generate | finalize)` retries automatically on compile errors, runtime errors, or failing test cases
- The `should_continue` router checks both a `success` flag and an `iteration` counter against `max_iterations` to guarantee termination

### 3. Iterative Research Refinement
- The same cyclic shape can drive depth-based refinement instead of error correction: `research -> generate_questions -> (loop back to research | synthesize)`
- Each round's follow-up question steers the next round's research query, producing progressively deeper findings

### Next Steps
- Continue to the next notebook in `03_LangGraph_Fundamentals` to explore additional graph patterns
- Experiment with raising `max_iterations` / `max_depth` and observe how the loop behaves as the guard condition changes